# Persistence hidden-state analysis (Approach B)

**Objective.** For the 10 competent variable-timing circular checkpoints, compare internal (hidden-state) dynamics under three carried-state persistence settings:

| Persistence | Role |
|---|---|
| **1.00** | Native baseline (operator is a no-op at 1.00) |
| **0.95** | Candidate strength |
| **0.90** | Stronger neighbour |

**Seeds.** `20260801–20260806`, `20260808–20260811` (10 retained circular checkpoints).

**Outputs.** Per seed and pooled: (1) clean-delay drift + step-speed/norm metrics, (2) onset-aligned distractor attraction + recovery, (3) frozen-basis PCA trajectories of mean delay-window dynamics. All conditions reuse the **same frozen banks** (paired target angles, distractor angles, and per-trial distractor onsets).

**Claim boundary.** Descriptive mechanism evidence only. **No** matched-cost Gaussian comparator and **no** specificity claim vs noise.

In [1]:
import hashlib
import json
import pathlib
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from dataclasses import replace
from sklearn.decomposition import PCA

# Ensure repo root is cwd and src is importable (nbconvert may start in notebooks/)
REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / "src" / "wm_rnn").is_dir():
    candidate = REPO_ROOT.parent if REPO_ROOT.name == "notebooks" else REPO_ROOT
    if (candidate / "src" / "wm_rnn").is_dir():
        REPO_ROOT = candidate
    else:
        raise RuntimeError(f"Run this notebook from the worktree root; cwd={pathlib.Path.cwd()}")
if pathlib.Path.cwd().resolve() != REPO_ROOT.resolve():
    import os

    os.chdir(REPO_ROOT)
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from wm_rnn.config import load_config
from wm_rnn.hidden_angle_decoder import decode_angles_from_hidden
from wm_rnn.perturbation_experiment import (
    FINAL_SEED_BASE,
    _collect_batches,
    _git_metadata,
    _load_checkpoint_model,
    _operator_forward,
    fit_frozen_decoder,
)
from wm_rnn.perturbation_metrics import (
    _signed_wrapped_radians,
    activation_slope_and_saturation,
    delay_decoding_error,
    distractor_drift_and_recovery,
    signed_circular_error,
)
from wm_rnn.full_candidate_perturbation_run import trained_distractor_checkpoints
from wm_rnn.state_persistence_dense_run import (
    BASE_CONFIG,
    BASE_CONFIG_SHA256,
    CONFIG,
    POOL_MANIFEST,
)
from wm_rnn.training_utils import task_config_from_dict

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={DEVICE}, cuda_available={torch.cuda.is_available()}")
print(f"repo_root={REPO_ROOT}")
print(f"FINAL_SEED_BASE={FINAL_SEED_BASE}")
print(f"CONFIG={CONFIG}, BASE_CONFIG={BASE_CONFIG}")
print(f"POOL_MANIFEST={POOL_MANIFEST}")


device=cuda, cuda_available=True
repo_root=C:\Users\Bob Rice\Documents\Obsidian Vault\Dissertation\working-memory-rnn\.worktrees\variable-timing-circular
FINAL_SEED_BASE=202607300
CONFIG=configs\state_persistence_dense_variable_timing_1024.yaml, BASE_CONFIG=configs\fixation_circular_variable_distractor_working_memory.yaml
POOL_MANIFEST=outputs\fixation_circular_variable_distractor_working_memory\metrics\fixation_circular_variable_distractor_working_memory_pool_summary.json


In [2]:
PERSISTENCE = (1.00, 0.95, 0.90)
FAMILY = "A"
CLEAN_DELAY = 20
DISTRACTOR_DELAY = 20
DISTRACTOR_STEPS = 5
TRIALS = 1024
BATCH_SIZE = 128
N_BATCHES = TRIALS // BATCH_SIZE
assert TRIALS % BATCH_SIZE == 0

OUT = pathlib.Path("outputs/persistence_hidden_state_analysis")
METRICS_DIR = OUT / "metrics"
FIGURES_DIR = OUT / "figures"
METRICS_CSV = METRICS_DIR / "persistence_hidden_state_metrics.csv"
METRICS_JSON = METRICS_DIR / "persistence_hidden_state_summary.json"
FIG_PCA = FIGURES_DIR / "persistence_pca_delay_trajectories.png"
FIG_DRIFT = FIGURES_DIR / "persistence_drift_recovery_summary.png"
RESULTS_NOTE = pathlib.Path(
    "docs/reports/persistence_hidden_state_analysis_results.md"
)

METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_NOTE.parent.mkdir(parents=True, exist_ok=True)

EXPECTED_SEEDS = (
    20260801,
    20260802,
    20260803,
    20260804,
    20260805,
    20260806,
    20260808,
    20260809,
    20260810,
    20260811,
)

print(
    f"PERSISTENCE={PERSISTENCE}, TRIALS={TRIALS}, "
    f"BATCH_SIZE={BATCH_SIZE}, N_BATCHES={N_BATCHES}"
)
print(f"OUT={OUT.resolve()}")


PERSISTENCE=(1.0, 0.95, 0.9), TRIALS=1024, BATCH_SIZE=128, N_BATCHES=8
OUT=C:\Users\Bob Rice\Documents\Obsidian Vault\Dissertation\working-memory-rnn\.worktrees\variable-timing-circular\outputs\persistence_hidden_state_analysis


In [3]:
def _sha256(path: pathlib.Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest().upper()


base_config = load_config(BASE_CONFIG)
observed_sha = _sha256(pathlib.Path(BASE_CONFIG))
assert (
    observed_sha == BASE_CONFIG_SHA256
), f"base config SHA256 mismatch: {observed_sha} != {BASE_CONFIG_SHA256}"

task_config = replace(
    task_config_from_dict(base_config),
    batch_size=BATCH_SIZE,
    distractor_steps=DISTRACTOR_STEPS,
)

checkpoints = trained_distractor_checkpoints(".", POOL_MANIFEST)
ckpt_seeds = tuple(int(ckpt.seed) for ckpt in checkpoints)
assert ckpt_seeds == EXPECTED_SEEDS, f"unexpected seeds: {ckpt_seeds}"
assert len(checkpoints) == 10

print(f"base_config_sha256={observed_sha}")
print(f"task_config.batch_size={task_config.batch_size}")
print(f"task_config.distractor_steps={task_config.distractor_steps}")
print(f"checkpoints ({len(checkpoints)}): {ckpt_seeds}")
for ckpt in checkpoints:
    print(f"  seed={ckpt.seed} path={ckpt.path} sha256={ckpt.sha256[:12]}...")


base_config_sha256=C568B49FBF17504D6047454E150C00C54F3E8C9503CE9E4EDD50C2CDA5FA554D
task_config.batch_size=128
task_config.distractor_steps=5
checkpoints (10): (20260801, 20260802, 20260803, 20260804, 20260805, 20260806, 20260808, 20260809, 20260810, 20260811)
  seed=20260801 path=outputs\fixation_circular_variable_distractor_working_memory\seed_sweep\seed_20260801\checkpoints\fixation_circular_variable_distractor_working_memory_seed_20260801.pt sha256=466DFF189524...
  seed=20260802 path=outputs\fixation_circular_variable_distractor_working_memory\seed_sweep\seed_20260802\checkpoints\fixation_circular_variable_distractor_working_memory_seed_20260802.pt sha256=D7F6BDB3F62B...
  seed=20260803 path=outputs\fixation_circular_variable_distractor_working_memory\seed_sweep\seed_20260803\checkpoints\fixation_circular_variable_distractor_working_memory_seed_20260803.pt sha256=1A20AFA1E90F...
  seed=20260804 path=outputs\fixation_circular_variable_distractor_working_memory\seed_sweep\seed_20260

## Collection helpers

Two thin wrappers call `perturbation_experiment._collect_batches` so baseline and perturbed conditions share the same frozen banks:

- **`collect_clean`** — Family A clean trials, `condition_index=0`, fixed delay 20, `randomize_distractor_onsets=False`. Persistence `1.00` uses `forward_fn=None` (native); other gains use `state_persistence` via `_operator_forward`.
- **`collect_distractor`** — Family A distractor trials, `condition_index=1`, `randomize_distractor_onsets=True` so per-trial onsets come from the balanced bank.

**GAP-1 mitigation.** Stock `distractor_drift_and_recovery` assumes one shared distractor/post window. With variable onsets we align each trial to its own onset before averaging (`onset_aligned_attraction` below). Attraction/arc math matches `perturbation_metrics` (`_signed_wrapped_radians` displacement / target→distractor arc). Recovery is averaged across onset buckets that retain a post-distractor in-delay window; buckets with zero post steps are flagged and dropped from recovery only.

In [4]:
def collect_clean(model, task_config, gain: float):
    """Collect clean delay trials under native or state_persistence forward."""
    if float(gain) == 1.0:
        forward_fn = None
    else:
        forward_fn = _operator_forward(
            model,
            task_config,
            operator="state_persistence",
            variant="carried_state_only",
            strength=float(gain),
            condition="clean",
            family=FAMILY,
            delay_steps=CLEAN_DELAY,
        )
    return _collect_batches(
        model,
        task_config,
        FAMILY,
        "clean",
        0,
        CLEAN_DELAY,
        seed_base=FINAL_SEED_BASE,
        n_batches=N_BATCHES,
        batch_size=BATCH_SIZE,
        forward_fn=forward_fn,
        randomize_distractor_onsets=False,
    )


In [5]:
def collect_distractor(model, task_config, gain: float):
    """Collect distractor trials with frozen randomized per-trial onsets."""
    if float(gain) == 1.0:
        forward_fn = None
    else:
        forward_fn = _operator_forward(
            model,
            task_config,
            operator="state_persistence",
            variant="carried_state_only",
            strength=float(gain),
            condition="distractor",
            family=FAMILY,
            delay_steps=DISTRACTOR_DELAY,
            randomize_distractor_onsets=True,
        )
    collected = _collect_batches(
        model,
        task_config,
        FAMILY,
        "distractor",
        1,
        DISTRACTOR_DELAY,
        seed_base=FINAL_SEED_BASE,
        n_batches=N_BATCHES,
        batch_size=BATCH_SIZE,
        forward_fn=forward_fn,
        randomize_distractor_onsets=True,
    )
    assert collected["distractor_relative_starts"] is not None
    assert collected["distractor_angles"] is not None
    return collected


In [6]:
def onset_aligned_attraction(
    decoded: np.ndarray,
    targets: np.ndarray,
    distractors: np.ndarray,
    relative_starts: np.ndarray,
    delay_start: int,
    duration: int,
    delay_stop: int,
    post_len_min: int = 1,
) -> dict[str, float]:
    """Onset-aligned distractor attraction/recovery (GAP-1 notebook helper).

    Peak attraction and peak drift use the aligned distractor window over all
    trials. Recovery is computed per onset bucket that retains
    ``avail_post >= post_len_min`` in-delay steps after the distractor, then
    averaged across retained buckets. Buckets with insufficient post window are
    dropped from recovery only (never padded).
    """
    decoded = np.asarray(decoded, dtype=np.float64)
    targets = np.asarray(targets, dtype=np.float64)
    distractors = np.asarray(distractors, dtype=np.float64)
    relative_starts = np.asarray(relative_starts, dtype=np.int64)
    if decoded.ndim != 2:
        raise ValueError("decoded must have shape [time, trials]")
    n_trials = decoded.shape[1]
    if targets.shape != (n_trials,) or distractors.shape != (n_trials,):
        raise ValueError("targets/distractors must have shape [trials]")
    if relative_starts.shape != (n_trials,):
        raise ValueError("relative_starts must have shape [trials]")

    distractor_arc = _signed_wrapped_radians(distractors - targets)
    if np.any(np.isclose(distractor_arc, 0.0, atol=1e-12)):
        raise ValueError("target and distractor angles must differ on every trial")

    during_attr = np.empty((duration, n_trials), dtype=np.float64)
    during_abs_err_deg = np.empty((duration, n_trials), dtype=np.float64)
    for trial_idx, rel in enumerate(relative_starts):
        t0 = int(delay_start + rel)
        t1 = t0 + duration
        if t1 > delay_stop or t1 > decoded.shape[0]:
            raise ValueError(
                f"trial {trial_idx} distractor window [{t0},{t1}) exceeds delay"
            )
        traj = decoded[t0:t1, trial_idx]
        displacement = _signed_wrapped_radians(traj - targets[trial_idx])
        during_attr[:, trial_idx] = displacement / distractor_arc[trial_idx]
        during_abs_err_deg[:, trial_idx] = np.degrees(np.abs(displacement))

    mean_during = np.mean(during_attr, axis=1)
    peak_index = int(np.argmax(np.abs(mean_during)))
    peak_attraction = float(mean_during[peak_index])
    peak_drift_degrees = float(np.max(np.mean(during_abs_err_deg, axis=1)))

    recovery_by_onset: list[float] = []
    end_by_onset: list[float] = []
    dropped_onsets: list[int] = []
    retained_onsets: list[int] = []
    for onset in sorted(np.unique(relative_starts)):
        mask = relative_starts == onset
        avail_post = int(delay_stop - (delay_start + int(onset) + duration))
        if avail_post < post_len_min:
            dropped_onsets.append(int(onset))
            continue
        retained_onsets.append(int(onset))
        trial_ids = np.flatnonzero(mask)
        # Align: distractor duration + this onset's in-delay post window
        aligned = np.empty((duration + avail_post, trial_ids.size), dtype=np.float64)
        for col, trial_idx in enumerate(trial_ids):
            t0 = int(delay_start + onset)
            traj = decoded[t0 : t0 + duration + avail_post, trial_idx]
            displacement = _signed_wrapped_radians(traj - targets[trial_idx])
            aligned[:, col] = displacement / distractor_arc[trial_idx]
        mean_traj = np.mean(aligned, axis=1)
        peak_idx = int(np.argmax(np.abs(mean_traj)))
        bucket_peak = float(mean_traj[peak_idx])
        bucket_end = float(np.mean(aligned[-1]))
        end_by_onset.append(bucket_end)
        if np.isclose(bucket_peak, 0.0, atol=1e-12):
            recovery_by_onset.append(float("nan"))
        else:
            recovery_by_onset.append(
                float((bucket_peak - bucket_end) / bucket_peak)
            )

    end_attraction = (
        float(np.nanmean(end_by_onset)) if end_by_onset else float("nan")
    )
    recovery_fraction = (
        float(np.nanmean(recovery_by_onset)) if recovery_by_onset else float("nan")
    )

    return {
        "distractor_peak_attraction_fraction": peak_attraction,
        "distractor_end_attraction_fraction": end_attraction,
        "distractor_recovery_fraction": recovery_fraction,
        "distractor_peak_drift_degrees": peak_drift_degrees,
        "recovery_n_onset_buckets": float(len(retained_onsets)),
        "recovery_dropped_onset_buckets": float(len(dropped_onsets)),
        "recovery_dropped_onsets_max": (
            float(max(dropped_onsets)) if dropped_onsets else float("nan")
        ),
    }


print("onset_aligned_attraction defined")


onset_aligned_attraction defined


In [7]:
def _hidden_speed(hidden: np.ndarray) -> np.ndarray:
    """Local copy of baseline_competence_figures._hidden_speed."""
    deltas = hidden[1:] - hidden[:-1]
    return np.linalg.norm(deltas, axis=-1)


def clean_delay_metrics(collected: dict, weights: np.ndarray) -> dict[str, float]:
    """Clean-delay decode error, start→end drift, norm, speed, activation."""
    delay = collected["phase_index"]["delay"]
    hidden = collected["hidden_states"]
    angles = collected["angles"]

    de = delay_decoding_error(hidden, angles, weights, delay)
    decoded = decode_angles_from_hidden(hidden, weights)
    start_angles = decoded[delay.start]
    end_angles = decoded[delay.stop - 1]
    drift_degrees = float(
        np.mean(np.abs(signed_circular_error(end_angles, start_angles)))
    )

    delay_hidden = hidden[delay]
    mean_norm = float(np.mean(np.linalg.norm(delay_hidden, axis=-1)))
    speed = _hidden_speed(hidden)[delay.start : delay.stop - 1]
    mean_speed = float(np.mean(speed))
    act = activation_slope_and_saturation(delay_hidden)

    return {
        "clean_mean_error_degrees": float(de["mean_error_degrees"]),
        "clean_median_error_degrees": float(de["median_error_degrees"]),
        "clean_start_end_drift_degrees": drift_degrees,
        "clean_mean_hidden_norm": mean_norm,
        "clean_mean_step_speed": mean_speed,
        "clean_mean_activation_slope": float(act["mean_activation_slope"]),
        "clean_saturation_fraction": float(act["saturation_fraction"]),
    }


In [8]:
def distractor_metrics(collected: dict, weights: np.ndarray) -> dict[str, float]:
    """Onset-aligned distractor attraction and recovery scalars."""
    decoded = decode_angles_from_hidden(collected["hidden_states"], weights)
    delay = collected["phase_index"]["delay"]
    return onset_aligned_attraction(
        decoded,
        collected["angles"],
        collected["distractor_angles"],
        collected["distractor_relative_starts"],
        delay_start=int(delay.start),
        duration=DISTRACTOR_STEPS,
        delay_stop=int(delay.stop),
        post_len_min=1,
    )


## Sanity checks (pairing, no-op, shapes, onsets)

Run once on the first checkpoint before the full 10-seed loop.

In [9]:
sanity_ckpt = checkpoints[0]
sanity_model = _load_checkpoint_model(base_config, sanity_ckpt.path, DEVICE)
sanity_weights = fit_frozen_decoder(sanity_model, task_config, FAMILY)

# --- Pairing check: frozen banks identical across persistence ---
d_100 = collect_distractor(sanity_model, task_config, 1.00)
d_090 = collect_distractor(sanity_model, task_config, 0.90)
c_100 = collect_clean(sanity_model, task_config, 1.00)
c_090 = collect_clean(sanity_model, task_config, 0.90)

assert np.array_equal(d_100["angles"], d_090["angles"])
assert np.array_equal(d_100["distractor_angles"], d_090["distractor_angles"])
assert np.array_equal(
    d_100["distractor_relative_starts"], d_090["distractor_relative_starts"]
)
assert np.array_equal(c_100["angles"], c_090["angles"])
print("PAIRING OK: angles / distractor_angles / relative_starts match across gains")

# --- No-op check: persistence 1.00 operator == native forward_fn=None ---
noop_forward = _operator_forward(
    sanity_model,
    task_config,
    operator="state_persistence",
    variant="carried_state_only",
    strength=1.0,
    condition="clean",
    family=FAMILY,
    delay_steps=CLEAN_DELAY,
)
native = _collect_batches(
    sanity_model,
    task_config,
    FAMILY,
    "clean",
    0,
    CLEAN_DELAY,
    seed_base=FINAL_SEED_BASE,
    n_batches=1,
    batch_size=BATCH_SIZE,
    forward_fn=None,
    randomize_distractor_onsets=False,
)
persisted = _collect_batches(
    sanity_model,
    task_config,
    FAMILY,
    "clean",
    0,
    CLEAN_DELAY,
    seed_base=FINAL_SEED_BASE,
    n_batches=1,
    batch_size=BATCH_SIZE,
    forward_fn=noop_forward,
    randomize_distractor_onsets=False,
)
assert np.allclose(
    native["hidden_states"], persisted["hidden_states"], rtol=0.0, atol=1e-6
), "persistence 1.00 is not a no-op vs native"
print("NO-OP OK: persistence 1.00 hidden states allclose to native model(inputs)")

# --- Shape contracts ---
assert c_100["hidden_states"].shape == (c_100["hidden_states"].shape[0], TRIALS, 64)
assert sanity_weights.shape == (64, 2)
decoded_shape = decode_angles_from_hidden(c_100["hidden_states"], sanity_weights).shape
assert decoded_shape == (c_100["hidden_states"].shape[0], TRIALS)
print(
    f"SHAPES OK: hidden={c_100['hidden_states'].shape}, "
    f"weights={sanity_weights.shape}, decoded={decoded_shape}"
)

# --- Range check (tanh support) ---
h = c_100["hidden_states"]
assert h.min() >= -1.0 - 1e-6 and h.max() <= 1.0 + 1e-6
_ = activation_slope_and_saturation(h[c_100["phase_index"]["delay"]])
print(f"RANGE OK: hidden in [{h.min():.6f}, {h.max():.6f}]")

# --- Onset coverage ---
starts = d_100["distractor_relative_starts"]
counts = np.bincount(starts, minlength=16)
assert starts.shape == (TRIALS,)
assert int(counts.sum()) == TRIALS
assert counts.shape[0] >= 16
assert np.all(counts[:16] > 0), f"non-uniform / missing onsets: {counts[:16]}"
print(f"ONSET OK: bincount[0:16]={counts[:16].tolist()} sum={int(counts.sum())}")

del sanity_model
torch.cuda.empty_cache()
print("Sanity checks passed; GPU cache cleared.")


PAIRING OK: angles / distractor_angles / relative_starts match across gains
NO-OP OK: persistence 1.00 hidden states allclose to native model(inputs)
SHAPES OK: hidden=(90, 1024, 64), weights=(64, 2), decoded=(90, 1024)
RANGE OK: hidden in [-0.963296, 0.935518]
ONSET OK: bincount[0:16]=[64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64] sum=1024
Sanity checks passed; GPU cache cleared.


In [10]:
rows = []
pca_store = {}  # seed -> {gain -> mean delay trajectory [delay_len, units]}
baseline_delay_hidden = {}  # seed -> [delay_len, trials, units] for PCA fit
baseline_angles = {}
pca_objects = {}  # filled in next cell; placeholder for id checks

for ckpt_i, ckpt in enumerate(checkpoints):
    print(f"[{ckpt_i + 1}/{len(checkpoints)}] seed={ckpt.seed} loading…")
    model = _load_checkpoint_model(base_config, ckpt.path, DEVICE)
    weights = fit_frozen_decoder(model, task_config, FAMILY)
    assert weights.shape == (64, 2)

    for gain in PERSISTENCE:
        print(f"  persistence={gain:.2f} collecting…")
        clean = collect_clean(model, task_config, gain)
        dist = collect_distractor(model, task_config, gain)

        assert clean["hidden_states"].shape[1] == TRIALS
        assert dist["hidden_states"].shape[1] == TRIALS
        assert clean["hidden_states"].shape[2] == 64

        row = {
            "seed": int(ckpt.seed),
            "persistence": float(gain),
            **clean_delay_metrics(clean, weights),
            **distractor_metrics(dist, weights),
        }
        rows.append(row)

        delay = clean["phase_index"]["delay"]
        pca_store.setdefault(ckpt.seed, {})[gain] = clean["hidden_states"][
            delay
        ].mean(axis=1)
        if float(gain) == 1.0:
            baseline_delay_hidden[ckpt.seed] = np.array(
                clean["hidden_states"][delay], copy=True
            )
            baseline_angles[ckpt.seed] = np.array(clean["angles"], copy=True)

        del clean, dist

    del model
    torch.cuda.empty_cache()
    print(f"  seed={ckpt.seed} done; GPU cache cleared")

metrics_df = pd.DataFrame(rows)
assert len(metrics_df) == 30, f"expected 30 rows, got {len(metrics_df)}"
assert set(metrics_df["seed"]) == set(EXPECTED_SEEDS)
assert set(metrics_df["persistence"]) == set(PERSISTENCE)
print(metrics_df.round(4).to_string(index=False))


[1/10] seed=20260801 loading…


  persistence=1.00 collecting…


  persistence=0.95 collecting…


  persistence=0.90 collecting…


  seed=20260801 done; GPU cache cleared
[2/10] seed=20260802 loading…


...

  seed=20260810 done; GPU cache cleared
[10/10] seed=20260811 loading…


  persistence=1.00 collecting…


  persistence=0.95 collecting…


  persistence=0.90 collecting…


  seed=20260811 done; GPU cache cleared
    seed  persistence  clean_mean_error_degrees  clean_median_error_degrees  clean_start_end_drift_degrees  clean_mean_hidden_norm  clean_mean_step_speed  clean_mean_activation_slope  clean_saturation_fraction  distractor_peak_attraction_fraction  distractor_end_attraction_fraction  distractor_recovery_fraction  distractor_peak_drift_degrees  recovery_n_onset_buckets  recovery_dropped_onset_buckets  recovery_dropped_onsets_max
20260801         1.00                    0.6923                      0.5565                         1.0467                  5.3577                 0.1214                       0.5513                     0.0015                               0.0842                              0.0426                        0.3959                         3.8111                      15.0                             1.0                         15.0
20260801         0.95                    0.6388                      0.5058                       

In [11]:
# Frozen per-seed PCA: fit once on baseline (1.00) delay-window tanh states; project means
projected_means = {}  # seed -> {gain -> [delay_len, 2]}
explained_ratios = {}

for seed in EXPECTED_SEEDS:
    X = baseline_delay_hidden[seed].reshape(-1, baseline_delay_hidden[seed].shape[-1])
    pca = PCA(n_components=2).fit(X)  # tanh states directly; no arctanh
    pca_objects[seed] = pca
    explained_ratios[seed] = pca.explained_variance_ratio_.copy()
    projected_means[seed] = {}
    pca_id = id(pca)
    for gain in PERSISTENCE:
        mean_traj = pca_store[seed][gain]
        projected = pca.transform(mean_traj)
        projected_means[seed][gain] = projected
        assert id(pca) == pca_id, "PCA object mutated / refit across gains"

rep_seed = EXPECTED_SEEDS[0]
print(
    f"Representative seed {rep_seed}: "
    f"explained_var={explained_ratios[rep_seed]}, "
    f"sum={explained_ratios[rep_seed][:2].sum():.4f}"
)
print(
    "PCA SANITY OK: per-seed frozen basis; "
    f"rep PC1+PC2={100 * explained_ratios[rep_seed][:2].sum():.1f}%"
)


Representative seed 20260801: explained_var=[0.46477136 0.44467443], sum=0.9094
PCA SANITY OK: per-seed frozen basis; rep PC1+PC2=90.9%


In [12]:
# Pooled summary mean ± SD across seeds per persistence
metric_cols = [
    c
    for c in metrics_df.columns
    if c not in ("seed", "persistence")
]
summary = {
    "provenance": {
        "base_config": str(BASE_CONFIG),
        "base_config_sha256": observed_sha,
        "dense_config": str(CONFIG),
        "pool_manifest": str(POOL_MANIFEST),
        "final_seed_base": FINAL_SEED_BASE,
        "family": FAMILY,
        "trials_per_condition": TRIALS,
        "batch_size": BATCH_SIZE,
        "persistence_values": list(PERSISTENCE),
        "checkpoint_seeds": list(EXPECTED_SEEDS),
        "checkpoint_sha256": {
            int(ckpt.seed): ckpt.sha256 for ckpt in checkpoints
        },
        **_git_metadata(),
        "claim_boundary": (
            "Descriptive hidden-state mechanism evidence for state_persistence "
            "on the variable-timing circular family only. Not a matched-cost "
            "specificity claim vs Gaussian noise; no confirmatory inference."
        ),
        "distractor_alignment": (
            "Onset-aligned attraction over the distractor window for all trials; "
            "recovery averaged across onset buckets with avail_post >= 1 "
            "(late onsets with zero in-delay post window dropped from recovery only)."
        ),
    },
    "pooled_by_persistence": {},
}

for gain in PERSISTENCE:
    sub = metrics_df[np.isclose(metrics_df["persistence"], gain)]
    pooled = {}
    for col in metric_cols:
        values = sub[col].to_numpy(dtype=np.float64)
        pooled[col] = {
            "mean": float(np.nanmean(values)),
            "std": float(np.nanstd(values, ddof=0)),
            "n": int(np.sum(np.isfinite(values))),
        }
    summary["pooled_by_persistence"][f"{gain:.2f}"] = pooled

metrics_df.to_csv(METRICS_CSV, index=False)
with METRICS_JSON.open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2)

print(f"wrote {METRICS_CSV} ({len(metrics_df)} rows)")
print(f"wrote {METRICS_JSON}")

# Descriptive monotonicity spot-check (not a gate)
for col in (
    "clean_mean_error_degrees",
    "clean_start_end_drift_degrees",
    "distractor_peak_attraction_fraction",
):
    means = [
        summary["pooled_by_persistence"][f"{g:.2f}"][col]["mean"]
        for g in PERSISTENCE
    ]
    print(f"spot-check {col}: " + ", ".join(f"{g:.2f}→{m:.4f}" for g, m in zip(PERSISTENCE, means)))


wrote outputs\persistence_hidden_state_analysis\metrics\persistence_hidden_state_metrics.csv (30 rows)
wrote outputs\persistence_hidden_state_analysis\metrics\persistence_hidden_state_summary.json
spot-check clean_mean_error_degrees: 1.00→0.9752, 0.95→1.2503, 0.90→1.9513
spot-check clean_start_end_drift_degrees: 1.00→1.6339, 0.95→1.7568, 0.90→2.2215
spot-check distractor_peak_attraction_fraction: 1.00→0.0845, 0.95→0.0944, 0.90→0.1058


In [13]:
# Figure 1: frozen-PCA mean delay trajectories for representative seed
plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.facecolor": "white",
        "savefig.facecolor": "white",
    }
)
COLOURS = {
    "blue": "#315A8C",
    "orange": "#D9822B",
    "green": "#3C8D6B",
    "red": "#C94C4C",
    "grey": "#6B7280",
}

rep_seed = EXPECTED_SEEDS[0]
ev = explained_ratios[rep_seed]
styles = {1.00: ("-", COLOURS["blue"]), 0.95: ("--", COLOURS["orange"]), 0.90: (":", COLOURS["red"])}
labels = {1.00: "persistence 1.00", 0.95: "persistence 0.95", 0.90: "persistence 0.90"}

fig, axes = plt.subplots(1, 2, figsize=(6.4, 5.0))

# Optional context: faint baseline single-trial trajectories coloured by cue angle
ax0 = axes[0]
pca = pca_objects[rep_seed]
base_h = baseline_delay_hidden[rep_seed]  # [delay, trials, units]
base_ang = baseline_angles[rep_seed]
n_show = min(48, base_h.shape[1])
cmap = plt.get_cmap("hsv")
for trial_idx in range(n_show):
    colour = cmap((base_ang[trial_idx] % (2 * np.pi)) / (2 * np.pi))
    traj = pca.transform(base_h[:, trial_idx, :])
    ax0.plot(traj[:, 0], traj[:, 1], color=colour, alpha=0.25, lw=0.8)
ax0.set_xlabel(f"PC1 ({100 * ev[0]:.1f}% var)")
ax0.set_ylabel(f"PC2 ({100 * ev[1]:.1f}% var)")
ax0.set_title(f"Baseline trials (seed {rep_seed})")
ax0.set_aspect("equal", adjustable="datalim")

ax1 = axes[1]
for gain in PERSISTENCE:
    traj = projected_means[rep_seed][gain]
    ls, colour = styles[gain]
    ax1.plot(traj[:, 0], traj[:, 1], ls=ls, color=colour, lw=2.0, label=labels[gain])
    ax1.scatter(traj[0, 0], traj[0, 1], facecolors="none", edgecolors=colour, s=55, zorder=3)
    ax1.scatter(traj[-1, 0], traj[-1, 1], color=colour, s=55, zorder=3)
ax1.set_xlabel(f"PC1 ({100 * ev[0]:.1f}% var)")
ax1.set_ylabel(f"PC2 ({100 * ev[1]:.1f}% var)")
ax1.set_title(f"Mean delay traj. (frozen PCA, seed {rep_seed})")
ax1.legend(frameon=False, fontsize=8)
ax1.set_aspect("equal", adjustable="datalim")

fig.tight_layout()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_PCA, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"wrote {FIG_PCA} ({FIG_PCA.stat().st_size} bytes)")


wrote outputs\persistence_hidden_state_analysis\figures\persistence_pca_delay_trajectories.png (101544 bytes)


In [14]:
# Figure 2: drift / recovery summary across persistence
fig, axes = plt.subplots(1, 2, figsize=(6.4, 4.2))
x = np.array(PERSISTENCE)

def _seed_overlay(ax, column, colour):
    for seed in EXPECTED_SEEDS:
        sub = metrics_df[metrics_df["seed"] == seed].sort_values("persistence")
        ax.plot(
            sub["persistence"],
            sub[column],
            "o",
            color=colour,
            alpha=0.35,
            ms=4,
            zorder=2,
        )


# Left: clean decode error + step speed
ax = axes[0]
clean_means = [
    summary["pooled_by_persistence"][f"{g:.2f}"]["clean_mean_error_degrees"]["mean"]
    for g in PERSISTENCE
]
clean_stds = [
    summary["pooled_by_persistence"][f"{g:.2f}"]["clean_mean_error_degrees"]["std"]
    for g in PERSISTENCE
]
speed_means = [
    summary["pooled_by_persistence"][f"{g:.2f}"]["clean_mean_step_speed"]["mean"]
    for g in PERSISTENCE
]
speed_stds = [
    summary["pooled_by_persistence"][f"{g:.2f}"]["clean_mean_step_speed"]["std"]
    for g in PERSISTENCE
]
_seed_overlay(ax, "clean_mean_error_degrees", COLOURS["blue"])
ax.errorbar(
    x,
    clean_means,
    yerr=clean_stds,
    fmt="o-",
    color=COLOURS["blue"],
    lw=1.6,
    capsize=3,
    label="clean decode error (°)",
)
ax2 = ax.twinx()
_seed_overlay(ax2, "clean_mean_step_speed", COLOURS["orange"])
ax2.errorbar(
    x,
    speed_means,
    yerr=speed_stds,
    fmt="s--",
    color=COLOURS["orange"],
    lw=1.4,
    capsize=3,
    label="step speed",
)
ax.set_xlabel("Persistence")
ax.set_ylabel("Clean delay decode error (°)", color=COLOURS["blue"])
ax2.set_ylabel("Mean delay step speed", color=COLOURS["orange"])
ax.set_xticks(list(PERSISTENCE))
ax.set_title("Clean delay dynamics")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, frameon=False, fontsize=7, loc="best")

# Right: distractor peak attraction + recovery
ax = axes[1]
attr_means = [
    summary["pooled_by_persistence"][f"{g:.2f}"][
        "distractor_peak_attraction_fraction"
    ]["mean"]
    for g in PERSISTENCE
]
attr_stds = [
    summary["pooled_by_persistence"][f"{g:.2f}"][
        "distractor_peak_attraction_fraction"
    ]["std"]
    for g in PERSISTENCE
]
rec_means = [
    summary["pooled_by_persistence"][f"{g:.2f}"]["distractor_recovery_fraction"][
        "mean"
    ]
    for g in PERSISTENCE
]
rec_stds = [
    summary["pooled_by_persistence"][f"{g:.2f}"]["distractor_recovery_fraction"][
        "std"
    ]
    for g in PERSISTENCE
]
_seed_overlay(ax, "distractor_peak_attraction_fraction", COLOURS["red"])
ax.errorbar(
    x,
    attr_means,
    yerr=attr_stds,
    fmt="o-",
    color=COLOURS["red"],
    lw=1.6,
    capsize=3,
    label="peak attraction",
)
_seed_overlay(ax, "distractor_recovery_fraction", COLOURS["green"])
ax.errorbar(
    x,
    rec_means,
    yerr=rec_stds,
    fmt="s--",
    color=COLOURS["green"],
    lw=1.4,
    capsize=3,
    label="recovery fraction",
)
ax.set_xlabel("Persistence")
ax.set_ylabel("Fraction")
ax.set_xticks(list(PERSISTENCE))
ax.set_title("Onset-aligned distractor")
ax.legend(frameon=False, fontsize=7)

fig.tight_layout()
fig.savefig(FIG_DRIFT, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"wrote {FIG_DRIFT} ({FIG_DRIFT.stat().st_size} bytes)")


wrote outputs\persistence_hidden_state_analysis\figures\persistence_drift_recovery_summary.png (130166 bytes)


In [15]:
# Write short results note with the exact pooled numbers
lines = [
    "# Persistence hidden-state analysis results",
    "",
    "Descriptive Approach-B analysis of `state_persistence` on the variable-timing",
    "circular family (10 seeds × 3 persistence values × 1,024 paired trials).",
    "**Claim boundary:** mechanism description only; no matched-cost Gaussian comparator.",
    "",
    "## Design",
    "",
    f"- Seeds: `{list(EXPECTED_SEEDS)}`",
    f"- Persistence: `{list(PERSISTENCE)}`",
    f"- Base config SHA256: `{observed_sha}`",
    f"- `FINAL_SEED_BASE`: `{FINAL_SEED_BASE}`",
    "- Distractor metric: onset-aligned; recovery averaged over onset buckets with",
    "  in-delay post window ≥ 1 step (late onsets dropped from recovery only).",
    "- PCA: per-seed basis fit on persistence-1.00 clean delay tanh states (no arctanh);",
    "  0.95/0.90 mean trajectories projected into that frozen basis.",
    "",
    "## Pooled means (across 10 seeds)",
    "",
    "| Persistence | Mean clean drift (°) | Mean clean decode err (°) | Mean step speed | Peak attraction | Recovery fraction |",
    "|---|---:|---:|---:|---:|---:|",
]
for gain in PERSISTENCE:
    p = summary["pooled_by_persistence"][f"{gain:.2f}"]
    lines.append(
        "| "
        f"{gain:.2f} | "
        f"{p['clean_start_end_drift_degrees']['mean']:.4f} | "
        f"{p['clean_mean_error_degrees']['mean']:.4f} | "
        f"{p['clean_mean_step_speed']['mean']:.4f} | "
        f"{p['distractor_peak_attraction_fraction']['mean']:.4f} | "
        f"{p['distractor_recovery_fraction']['mean']:.4f} |"
    )

lines.extend(
    [
        "",
        "## Artifacts",
        "",
        f"- Metrics CSV: `{METRICS_CSV.as_posix()}` ({len(metrics_df)} rows)",
        f"- Summary JSON: `{METRICS_JSON.as_posix()}`",
        f"- PCA figure: `{FIG_PCA.as_posix()}`",
        f"- Drift/recovery figure: `{FIG_DRIFT.as_posix()}`",
        "",
        "## Sanity checklist (executed in notebook)",
        "",
        "- Pairing: identical angles / distractor angles / relative starts across gains.",
        "- No-op: persistence 1.00 hidden states allclose to native `model(inputs)`.",
        "- Shapes: hidden `[T, 1024, 64]`, weights `[64, 2]`, decoded `[T, 1024]`.",
        "- Range: hidden within tanh support; onset bank covers 0..15 and sums to 1024.",
        "- PCA: frozen per-seed basis; object id unchanged across gains.",
        f"- Row count: {len(metrics_df)} (= 10 × 3).",
        "",
    ]
)
RESULTS_NOTE.write_text("\n".join(lines), encoding="utf-8")
print(f"wrote {RESULTS_NOTE}")
print("\n".join(lines[lines.index("## Pooled means (across 10 seeds)") :]))


wrote docs\reports\persistence_hidden_state_analysis_results.md
## Pooled means (across 10 seeds)

| Persistence | Mean clean drift (°) | Mean clean decode err (°) | Mean step speed | Peak attraction | Recovery fraction |
|---|---:|---:|---:|---:|---:|
| 1.00 | 1.6339 | 0.9752 | 0.0855 | 0.0845 | 0.3074 |
| 0.95 | 1.7568 | 1.2503 | 0.0787 | 0.0944 | 0.2925 |
| 0.90 | 2.2215 | 1.9513 | 0.0752 | 0.1058 | 0.2741 |

## Artifacts

- Metrics CSV: `outputs/persistence_hidden_state_analysis/metrics/persistence_hidden_state_metrics.csv` (30 rows)
- Summary JSON: `outputs/persistence_hidden_state_analysis/metrics/persistence_hidden_state_summary.json`
- PCA figure: `outputs/persistence_hidden_state_analysis/figures/persistence_pca_delay_trajectories.png`
- Drift/recovery figure: `outputs/persistence_hidden_state_analysis/figures/persistence_drift_recovery_summary.png`

## Sanity checklist (executed in notebook)

- Pairing: identical angles / distractor angles / relative starts across gains.
- No

## Results note & sanity checklist

Validation executed above:

1. **Pairing** — frozen banks identical across persistence for a fixed seed.
2. **No-op** — persistence 1.00 ≡ native forward on a small clean batch.
3. **Shapes** — hidden `[T, 1024, 64]`, decoder `[64, 2]`, decoded `[T, 1024]`.
4. **Range** — tanh states in `[-1, 1]`.
5. **Onset coverage** — balanced relative starts over `0..15`, sum 1024.
6. **PCA** — per-seed basis fit only on baseline; frozen for 0.95/0.90.
7. **Monotonicity** — printed as a descriptive spot-check only (not a gate).
8. **Row count** — exactly 30 metric rows.

Results note path: `docs/reports/persistence_hidden_state_analysis_results.md`.

**Claim boundary (restated).** This notebook is descriptive mechanism evidence for `state_persistence` on the variable-timing circular family. It does **not** introduce a matched-cost Gaussian comparator and does **not** support a specificity claim.